# Gold — fato_pedido_venda e dim_calendario
Fato no grão item do pedido (SC6 + SC5), sem registros excluídos, com saldo em carteira e situação do item.

In [ ]:
for nome, padrao in [("catalogo_silver", "dev_silver"), ("catalogo_gold", "dev_gold"), ("src_path", "")]:
    dbutils.widgets.text(nome, padrao)

import sys

src_path = dbutils.widgets.get("src_path")
if src_path and src_path not in sys.path:
    sys.path.append(src_path)

In [ ]:
from datetime import date

from pyspark.sql import functions as F

from pdc_lib.comentarios import comandos_comentario
from pdc_lib.transformacoes import montar_calendario, montar_fato
from pdc_lib.util import nome_tabela, validar_identificador

catalogo_silver = validar_identificador(dbutils.widgets.get("catalogo_silver"))
catalogo_gold = validar_identificador(dbutils.widgets.get("catalogo_gold"))

sc5_nome = nome_tabela(catalogo_silver, "protheus", "sc5")
sc6_nome = nome_tabela(catalogo_silver, "protheus", "sc6")
if not (spark.catalog.tableExists(sc5_nome) and spark.catalog.tableExists(sc6_nome)):
    dbutils.notebook.exit("Silver de pedidos ainda não disponível.")


def publicar(df, tabela: str) -> None:
    destino = nome_tabela(catalogo_gold, "comercial", tabela)
    df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(destino)
    for comando in comandos_comentario(destino, tabela):
        spark.sql(comando)
    print(f"{destino}: {df.count()} registros.")

In [ ]:
fato = montar_fato(spark.table(sc5_nome), spark.table(sc6_nome))
publicar(fato, "fato_pedido_venda")

inicio = fato.agg(F.min("dat_emissao")).first()[0] or date.today()
fim = date(max(date.today().year, inicio.year), 12, 31)
publicar(montar_calendario(spark, inicio.isoformat(), fim.isoformat()), "dim_calendario")